# ML training lab

To download this notebook click on this [link](https://github.com/SETU-AIML-2026/DH-Infra-AI/blob/main/topic-04-sprint-3-AI/unit-1-ai-training/notebook-1-svm-pytorch/lab7.ipynb)

Switch to T4 GPU Runtime:

    * In the top-right menu of this Colab page, click the arrow next to the Connect button (or click Runtime in the top menu bar).
    * Select Change runtime type.
    * Under Hardware accelerator, choose T4 GPU.
    * Click Save.

Accessing the Data:

    * Make sure you have a reliable internet connection so that the public URL (https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv) is accessible during training. See previous lab materials on how to split the data for training.
    * If you still have it, you can load it from Google Cloud Storage (GCS), ensure your GCS bucket permissions allow access, and authenticate using Google Cloud SDK tools in the notebook.



### Model Hyperparameter Configuration

In [ ]:
# Define hyperparameters for SVM and Deep Neural Network
SVM_RANDOM_STATE = 42
SVM_MAX_ITER = 1000

# PyTorch Neural Network Hyperparameters
NN_INPUT_SIZE = 8  # Number of preprocessed features (Pclass, Age, SibSp, Parch, Fare, Sex_male, Embarked_Q, Embarked_S)
NN_HIDDEN_SIZE = 1000
NN_EPOCHS = 100
NN_BATCH_SIZE = 64
NN_LEARNING_RATE = 0.001
NN_RANDOM_STATE = 42

# Device selection
DEVICE = 'cpu' #normally it would be torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

### Load Data, Train and Compare Models

# Imports
this part should be in a requirements.txt with exact version numbers!

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import warnings
import time
from google.cloud import storage
from google.colab import auth

warnings.filterwarnings('ignore')

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(NN_RANDOM_STATE)
np.random.seed(NN_RANDOM_STATE)

# Connecting to GCS
The project and bucket name should be in an enviromental variable

In [ ]:
# --- GCP Configuration ---
PROJECT_ID = "Dh-inf-for-AI"
BUCKET_NAME = "titanic-dh-data"

auth.authenticate_user()

In [ ]:
# Load the Titanic dataset splits from GCS sharded folder
train_url = f"gs://{BUCKET_NAME}/sharded/train.csv"
dev_url = f"gs://{BUCKET_NAME}/sharded/dev.csv"
test_url = f"gs://{BUCKET_NAME}/sharded/test.csv"

# Reading the files using pandas
df_train = pd.read_csv(train_url)
df_dev = pd.read_csv(dev_url)
df_test = pd.read_csv(test_url)

print(f"Loaded train shape: {df_train.shape}, dev shape: {df_dev.shape}, test shape: {df_test.shape}")

In [ ]:
# Preparing features
features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex', 'Embarked']

# Helper function to process each dataset split consistently
def preprocess_split(df_split):
    df_processed = df_split[features].copy()

    # Fill missing values using training set statistics or defaults
    df_processed['Age'].fillna(df_train['Age'].median(), inplace=True)
    df_processed['Fare'].fillna(df_train['Fare'].median(), inplace=True)
    df_processed['Embarked'].fillna(df_train['Embarked'].mode()[0], inplace=True)

    # Convert categorical features to numeric
    df_processed = pd.get_dummies(df_processed, columns=['Sex', 'Embarked'], drop_first=True)

    # Ensure both splits have the exact same columns after dummy encoding
    expected_columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']
    for col in expected_columns:
        if col not in df_processed.columns:
            df_processed[col] = 0
    return df_processed[expected_columns]

# Prepare features for both Train and Dev splits
X_train = preprocess_split(df_train)
y_train = df_train['Survived']

X_test = preprocess_split(df_dev)
y_test = df_dev['Survived']

print(f"Final shapes - X_train: {X_train.shape}, X_test (dev): {X_test.shape}")

In [ ]:
#Feature Normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# AI training

In [ ]:
# Define PyTorch MLP Model Architecture
class TitanicMLP(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(TitanicMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

# Helper function to train and evaluate PyTorch model with timing
def train_and_eval_pytorch(X_tr, y_tr, X_te, y_te, device):
    start_time = time.time()

    # Convert data to PyTorch Tensors and cast to float32
    X_tr_t = torch.FloatTensor(np.array(X_tr).astype(np.float32)).to(device)
    y_tr_t = torch.FloatTensor(np.array(y_tr).astype(np.float32)).unsqueeze(1).to(device)
    X_te_t = torch.FloatTensor(np.array(X_te).astype(np.float32)).to(device)

    dataset = TensorDataset(X_tr_t, y_tr_t)
    loader = DataLoader(dataset, batch_size=NN_BATCH_SIZE, shuffle=True)

    # Instantiate model, loss, and optimizer
    model = TitanicMLP(X_tr.shape[1], NN_HIDDEN_SIZE).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=NN_LEARNING_RATE)

    loss_history = []

    # Training loop
    model.train()
    for epoch in range(NN_EPOCHS):
        epoch_loss = 0.0
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch_x.size(0)
        loss_history.append(epoch_loss / len(X_tr))

    # Evaluation
    model.eval()
    with torch.no_grad():
        predictions = model(X_te_t).cpu().numpy()
        predictions = (predictions > 0.5).astype(int)
        acc = accuracy_score(y_te, predictions)

    duration = time.time() - start_time
    return acc, loss_history, model, duration

# Define classic SVM (Unscaled vs Scaled)
svm_unscaled = SVC(random_state=SVM_RANDOM_STATE, max_iter=SVM_MAX_ITER)
svm_scaled = SVC(random_state=SVM_RANDOM_STATE, max_iter=SVM_MAX_ITER)

# 5. Train and Evaluate WITHOUT Normalization
print("=== WITHOUT FEATURE NORMALIZATION ===")
start_svm = time.time()
svm_unscaled.fit(X_train, y_train)
duration_svm_unscaled = time.time() - start_svm
y_pred_svm_unscaled = svm_unscaled.predict(X_test)
acc_svm_unscaled = accuracy_score(y_test, y_pred_svm_unscaled)
print(f"SVM Accuracy (Unscaled): {acc_svm_unscaled:.4f} (Took {duration_svm_unscaled:.4f} seconds)")

acc_nn_unscaled, loss_history_unscaled, nn_unscaled, duration_nn_unscaled = train_and_eval_pytorch(
    X_train, y_train, X_test, y_test, DEVICE
)
print(f"NN Accuracy (Unscaled):  {acc_nn_unscaled:.4f} (Took {duration_nn_unscaled:.4f} seconds)")

# 6. Train and Evaluate WITH Normalization
print("\n=== WITH FEATURE NORMALIZATION ===")
start_svm_scaled = time.time()
svm_scaled.fit(X_train_scaled, y_train)
duration_svm_scaled = time.time() - start_svm_scaled
y_pred_svm_scaled = svm_scaled.predict(X_test_scaled)
acc_svm_scaled = accuracy_score(y_test, y_pred_svm_scaled)
print(f"SVM Accuracy (Scaled):   {acc_svm_scaled:.4f} (Took {duration_svm_scaled:.4f} seconds)")

acc_nn_scaled, loss_history_scaled, nn_scaled, duration_nn_scaled = train_and_eval_pytorch(
    X_train_scaled, y_train, X_test_scaled, y_test, DEVICE
)
print(f"NN Accuracy (Scaled):    {acc_nn_scaled:.4f} (Took {duration_nn_scaled:.4f} seconds)")

# Let's try with a GPU

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
print("\n=== WITH FEATURE NORMALIZATION AND GPU ===")
acc_nn_scaled_gpu, loss_history_scaled_gpu, nn_scaled_gpu, duration_nn_scaled_gpu = train_and_eval_pytorch(
    X_train_scaled, y_train, X_test_scaled, y_test, DEVICE
)
print(f"NN Accuracy (Scaled):    {acc_nn_scaled_gpu:.4f} (Took {duration_nn_scaled_gpu:.4f} seconds)")

### Demonstrating Feature Standardization (Mean = 0, Std = 1) for the Neural Network

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import time

# 1. Initialize the MinMaxScaler
min_max_scaler = MinMaxScaler()

# 2. Fit and transform the features to [0, 1]
X_train_normalized = min_max_scaler.fit_transform(X_train)
X_test_normalized = min_max_scaler.transform(X_test)

# 3. Initialize model for SVM
svm_normalized = SVC(random_state=SVM_RANDOM_STATE, max_iter=SVM_MAX_ITER)
start_svm_norm = time.time()
svm_normalized.fit(X_train_normalized, y_train)
duration_svm_normalized = time.time() - start_svm_norm
acc_svm_normalized = accuracy_score(y_test, svm_normalized.predict(X_test_normalized))

# 4. Train and evaluate PyTorch NN with MinMax Normalization
acc_nn_normalized, loss_history_normalized, nn_normalized, duration_nn_normalized = train_and_eval_pytorch(
    X_train_normalized, y_train, X_test_normalized, y_test, DEVICE
)

print("=== WITH MINMAX NORMALIZATION [0, 1] ===")
print(f"SVM Accuracy (MinMax Scaled): {acc_svm_normalized:.4f} (Took {duration_svm_normalized:.4f} seconds)")
print(f"NN Accuracy (MinMax Scaled):  {acc_nn_normalized:.4f} (Took {duration_nn_normalized:.4f} seconds)")

In [ ]:
import matplotlib.pyplot as plt

# Plotting the loss curve for the PyTorch Neural Network
plt.figure(figsize=(10, 6))
if 'loss_history_scaled' in locals():
    plt.plot(loss_history_scaled_gpu, label='Training Loss (Standardized Data)', color='blue', linewidth=2)

if 'loss_history_unscaled' in locals():
    plt.plot(loss_history_unscaled, label='Training Loss (Unscaled Data)', color='orange', linestyle='--', linewidth=2)

plt.title('PyTorch Neural Network Learning Curves', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import torch

# 1. Get predictions from the scaled PyTorch model
active_model = nn_scaled if 'nn_scaled' in locals() else nn_scaled_gpu
active_model.eval()

model_device = next(active_model.parameters()).device
X_test_t = torch.FloatTensor(np.array(X_test_scaled).astype(np.float32)).to(model_device)

with torch.no_grad():
    preds_prob = active_model(X_test_t).cpu().numpy().flatten()
    preds = (preds_prob > 0.5).astype(int)

# 2. Identify misclassified samples
test_targets = y_test.values
misclassified_mask = (preds != test_targets)

# 3. Use scaled features for plotting so features with different scales (e.g., Age vs Fare) can be easily compared together
features_list = list(X_train.columns)

# Convert overall scaled training features back to a DataFrame for plotting distributions
df_overall_scaled = pd.DataFrame(X_train_scaled, columns=features_list)

# Get the scaled features of only the misclassified test samples
df_misclassified_scaled = pd.DataFrame(X_test_scaled[misclassified_mask], columns=features_list)

# Melt overall data for boxplot representation
overall_melted = df_overall_scaled.melt(var_name='Feature', value_name='Standardized Value')

# Melt misclassified data for scatter overlay
misclassified_melted = df_misclassified_scaled.melt(var_name='Feature', value_name='Standardized Value')

# 4. Plot distributions and misclassified points
plt.figure(figsize=(14, 8))

# Create boxplots to show the overall distribution (median, quartiles, and std/outliers)
sns.boxplot(
    x='Feature',
    y='Standardized Value',
    data=overall_melted,
    color='lightblue',
    width=0.5,
    fliersize=0  # Hide default outliers to avoid confusion with red dots
)

# Overlay individual misclassified samples as simple red dots
sns.stripplot(
    x='Feature',
    y='Standardized Value',
    data=misclassified_melted,
    color='red',
    alpha=0.6,
    size=6,
    jitter=0.15,
    label='Misclassified Sample'
)

# Adjust legend to avoid duplicate labels for points
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend([list(by_label.values())[0]], [list(by_label.keys())[0]], loc='upper right')

plt.title('Misclassified Samples (Red Dots) vs. Overall Feature Distributions (Boxplots)', fontsize=16)
plt.xlabel('Features', fontsize=12)
plt.ylabel('Standardized Value (Mean=0, Std=1)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total Test Samples: {len(X_test_scaled)}")
print(f"Misclassified Samples plotted as red dots: {sum(misclassified_mask)}")

In [ ]:
# 1. Preprocess df_test
X_eval_test = preprocess_split(df_test)
y_eval_test = df_test['Survived']

# 2. Scale features for standardized models
X_eval_test_scaled = scaler.transform(X_eval_test)

print(f"Test set evaluation features shape: {X_eval_test.shape}")

# 3. Evaluate Unscaled SVM
y_pred_svm_test_unscaled = svm_unscaled.predict(X_eval_test)
acc_svm_test_unscaled = accuracy_score(y_eval_test, y_pred_svm_test_unscaled)
print(f"SVM Accuracy (Unscaled) on test set: {acc_svm_test_unscaled:.4f}")

# 4. Evaluate Scaled SVM
y_pred_svm_test_scaled = svm_scaled.predict(X_eval_test_scaled)
acc_svm_test_scaled = accuracy_score(y_eval_test, y_pred_svm_test_scaled)
print(f"SVM Accuracy (Scaled) on test set:   {acc_svm_test_scaled:.4f}")

# 5. Evaluate Neural Network on the CPU/GPU with scaled data
active_model = nn_scaled if 'nn_scaled' in locals() else nn_scaled_gpu
active_model.eval()
model_device = next(active_model.parameters()).device

X_eval_test_t = torch.FloatTensor(np.array(X_eval_test_scaled).astype(np.float32)).to(model_device)
with torch.no_grad():
    predictions_prob = active_model(X_eval_test_t).cpu().numpy()
    predictions = (predictions_prob > 0.5).astype(int).flatten()
    acc_nn_test_scaled = accuracy_score(y_eval_test, predictions)

print(f"NN Accuracy (Scaled) on test set:    {acc_nn_test_scaled:.4f}")

# Future Tasks

* Replace the current categorical feature handling with one-hot encoding.

* Apply standardisation or normalisation only to numerical features.

* Perform hyperparameter tuning to identify the best-performing configuration.

* Save and export the trained model to Google Cloud Storage (GCS), local storage, or Hugging Face.


